# config-mixin — Working with mixins

Mixins are bundles of Prometheus rules + Grafana dashboards rendered from Jsonnet. This notebook walks you through:

1. Listing every mixin defined across `/config/*.yaml`.
2. Adding a new mixin block from the community catalog.
3. Rendering one config end-to-end (`sync` → `init` → `render`) and counting the resulting dashboards / rule groups.
4. Browsing the catalog of well-known community mixins.

In [ ]:
CONFIG_DIR = '/config'
PICK_CONFIG = 'k8s-monitoring'   # one of the .yaml basenames in CONFIG_DIR

## 1. What's defined across all configs?

In [ ]:
import os, glob, yaml, pandas as pd

rows = []
for path in sorted(glob.glob(f'{CONFIG_DIR}/*.yaml')):
    cfg = yaml.safe_load(open(path)) or {}
    env = cfg.get('name', os.path.basename(path))
    for name, body in (cfg.get('mixins') or {}).items():
        src = (body or {}).get('source', {})
        kind = next(iter(src.keys()), '')
        url = src.get('git', {}).get('url') or src.get('directory', {}).get('path', '')
        rows.append({
            'env': env,
            'mixin': name,
            'source': kind,
            'url': url,
            'folder': (body.get('config') or {}).get('grafanaDashboardFolder', ''),
        })

df = pd.DataFrame(rows)
print(f'{len(df)} mixin entries across {df["env"].nunique()} environments')
df

## 2. Add a mixin

Drop a block like this under `mixins:` in your env config (e.g. `/config/k8s-monitoring.yaml`):

```yaml
mixins:
  argo-cd:
    source:
      git:
        url: https://github.com/adinhodovic/argo-cd-mixin.git
        ref: main
        depth: 1
    config:
      mimirNamespace: argo-cd
      grafanaDashboardFolder: Delivery
```

Then run `do-all` (or the focused commands in the next section) to vendor and render.

## 3. Render one config end-to-end

Runs `sync-mixins`, `init-mixins`, `render-resources`. No `apply-*`, so no credentials needed.

In [ ]:
import os, subprocess

env_name = PICK_CONFIG
cfg_path = f'{CONFIG_DIR}/{env_name}.yaml'
src_dir = f'/source/{env_name}'
build_dir = f'/build/{env_name}'

shenv = os.environ | {
    'CONFIG_FILE': cfg_path,
    'SOURCE_DIR': src_dir,
    'BUILD_DIR': src_dir,
}
for cmd in ['sync-mixins', 'init-mixins']:
    subprocess.run(['bash', '-lc', cmd], env=shenv, check=False)
subprocess.run(['bash', '-lc', 'render-resources'],
               env=shenv | {'BUILD_DIR': build_dir}, check=False)

In [ ]:
from collections import Counter

summary = Counter()
for root, _, files in os.walk(build_dir):
    bucket = os.path.relpath(root, build_dir)
    for f in files:
        summary[(bucket, f.rsplit('.', 1)[-1])] += 1

pd.DataFrame([{'bucket': b, 'ext': e, 'count': n} for (b, e), n in sorted(summary.items())])

## 4. Catalog of common community mixins

Documented in [`docs/mixins.md`](../docs/mixins.md). Drop the desired block into your env config and re-render.

In [ ]:
catalog = pd.DataFrame([
    ('Kubernetes',  'kubernetes-monitoring/kubernetes-mixin',     'Cluster-level monitoring (kubelet, cadvisor, apiserver, ...)'),
    ('Kubernetes',  'kubernetes/kube-state-metrics',              'State of Kubernetes objects'),
    ('Kubernetes',  'adinhodovic/ingress-nginx-mixin',            'NGINX Ingress Controller'),
    ('Kubernetes',  'imusmanmalik/cert-manager-mixin',            'Jetstack cert-manager'),
    ('Kubernetes',  'adinhodovic/argo-cd-mixin',                  'Argo CD'),
    ('Kubernetes',  'adinhodovic/kubernetes-events-mixin',        'Kubernetes events alerts/dashboards'),
    ('Kubernetes',  'adinhodovic/opencost-mixin',                 'OpenCost: per-namespace cost monitoring'),
    ('App / Infra', 'grafana/alloy',                              'Grafana Alloy collector'),
    ('App / Infra', 'adinhodovic/blackbox-exporter-mixin',        'Blackbox probes'),
    ('App / Infra', 'adinhodovic/django-mixin',                   'Django apps'),
    ('App / Infra', 'prometheus/node_exporter (docs/node-mixin)', 'Linux host metrics'),
    ('App / Infra', 'grafana/jsonnet-libs (windows-mixin)',       'Windows host metrics'),
], columns=['kind', 'source', 'description'])
catalog

### Where to go next

- For libraries that mixins depend on, see [`config-observ-lib.ipynb`](config-observ-lib.ipynb).
- For static `.json` dashboards from grafana.com or any HTTP URL, see [`config-grafana-dashboard.ipynb`](config-grafana-dashboard.ipynb).